In [1]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import rarfile
import tarfile
import py7zr
import pandas as pd

from thefuzz import process
import tqdm
from tqdm.auto import tqdm

In [2]:
def move_contents(src_dir, dst_dir, overwrite=True):
    for item in src_dir.iterdir():
        if item.is_file():
            item_name = item.name
            dst_path = dst_dir / item_name
            shutil.move(item, dst_path)
        elif item.is_dir():
            move_contents(item, dst_dir)

    src_dir_size = sum(file.stat().st_size for file in src_dir.rglob('*') if file.is_file())
    src_dir_empty = not any(src_dir.iterdir())
    if src_dir_empty or src_dir_size==0:
        shutil.rmtree(src_dir)

In [3]:
def is_base_zipfile(zip_path, filename='BasicFile.csv'):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        namelist = zf.namelist()
        return filename in namelist

In [4]:
def is_base_rar(rar_path, filename='BasicFile.csv'):
    with rarfile.RarFile(item) as rf:
        file_path_list = rf.namelist()
    file_list = []
    for item in file_path_list:
        file_list.append((Path(item).name))
    return filename in file_list

In [5]:
def rar_to_zip(rar_path, output_zip_path):
    extraction_dir = Path.cwd() / "ext_dir"
    with rarfile.RarFile(rar_path) as rf:
        rf.extractall(extraction_dir)
    base_dir_to_zip(extraction_dir, output_zip_path)

In [6]:
def is_base_dir(dir_path, filename='BasicFile.csv'):
    file_list = []
    for item in dir_path.iterdir():
        if item.is_file():
            file_list.append((Path(item).name))
    return filename in file_list

In [7]:
def move_to_base_dir(src_dir, dst_dir):
    for item in src_dir.iterdir():
        if item.suffix.lower().strip() == ".csv":
            item_name = item.name
            src_path = src_dir / item_name
            dst_path = dst_dir / item_name
            shutil.move(src_path, dst_path)

In [8]:
def aggregate_basic_zip(curr_path, parent_dir, irrelevant_files_dir, 
                        base_dir, processed_arch_dir, 
                        unprocessed_arch_dir, arch_extn):
    global global_counter
    # move all files to parent directory recursively
    for item in tqdm(parent_dir.rglob('*'), desc="Moving Files to parent_dir recursively..."):
        if item.exists():
            if item.is_file():
                if item.suffix.lower().strip() != ".csv":
                    dst_path = parent_dir / item.name
                    if dst_path.exists():
                        global_counter += 1
                        dst_path = parent_dir / f"{item.stem}_md{global_counter}{item.suffix}"
                    shutil.move(item, dst_path)
                else:
                    csv_parent = item.parent
                    print(csv_parent)
                    for file in csv_parent:
                        if file.suffix.lower().strip() == ".csv":
                            global_counter += 1
                            csv_dst_dir = base_dir / f"md{global_counter}"
                            os.makedirs(csv_dst_dir, exist_ok=True)
                            csv_dst_file = csv_dst_dir / file.name
                            shutil.move(file, csv_dst_file)

    # delete empty folders
    for item in tqdm(parent_dir.rglob('*'), desc="Handling directories..."):
        if item.is_dir():
            item_size = sum(file.stat().st_size for file in item.rglob('*') if file.is_file())
            item_empty = not any(item.iterdir())
            if item_empty or item_size==0:
                shutil.rmtree(item)
            
            
    
    #for item in tqdm(parent_dir.iterdir(), desc="Handling directories..."):
    #    if item.is_dir() and item.name != "00_new_folder":
    #        if is_base_dir(item):
    #            move_to_base_dir(item, base_dir)
    #            move_contents(item, parent_dir)
    #        else:
    #            move_contents(item, parent_dir)


    
        
    # now no directories are present at current location
    for item in tqdm(parent_dir.iterdir(), desc='Extracting archives...'):
        if item.is_file():
            # setting path to move the file after processing
            item_dir = item.parent
            item_name = item.name
            item_stem = item.stem    #item name without extension
            processed_arch = processed_arch_dir / item_name
            basic_zip_dest = base_dir / item_name
            unprocessed_arch = unprocessed_arch_dir / item_name
            irrelevant_file = irrelevant_files_dir / item_name
            
            if item.suffix in arch_extn:
                new_folder = parent_dir / "00_new_folder"
                os.makedirs(new_folder, exist_ok=True)
    
                try:
                    if zipfile.is_zipfile(item):
                        if is_base_zipfile(item):
                            shutil.move(item, basic_zip_dest)
                        else:
                            with zipfile.ZipFile(item, 'r') as zf:
                                zf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif rarfile.is_rarfile(item):
                        if is_base_rar(item):
                            item_name = f"{item_stem}.zip"
                            output_zip_path = item_dir / item_name
                            rar_to_zip(item, output_zip_path)
                            shutil.move(output_zip_path, basic_zip_dest)
                        else:
                            with rarfile.RarFile(item) as rf:
                                rf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif py7zr.is_7zfile(str(item)):
                        with py7zr.SevenZipFile(item, mode='r') as seven_zip:
                            seven_zip.extractall(path=new_folder)
                        shutil.move(item, processed_arch)
                        
                    elif tarfile.is_tarfile(item):
                        print(item)
                        with tarfile.open(item, 'r') as tar:
                            tar.extractall(new_folder)
                        shutil.move(item, processed_arch)
                        
                    else:
                        shutil.move(item, unprocessed_arch)
                        
                except Exception as e:
                    shutil.move(item, unprocessed_arch)
                    
            else:
                shutil.move(item, irrelevant_file)

    for item in parent_dir.iterdir():
        if item.is_dir() and item.name == "00_new_folder":
            aggregate_basic_zip(curr_path, item, irrelevant_files_dir, base_dir, processed_arch_dir, unprocessed_arch_dir, arch_extn)

In [8]:
def manage_files(curr_path, parent_dir, irrelevant_files_dir, 
                        base_dir, processed_arch_dir, 
                        unprocessed_arch_dir, arch_extn):
    
    for item in tqdm(parent_dir, desc="Processing..."):
        if item.is_file():
            item_extn = item.suffix.lower().strip()
            if item_extn == ".zip":
                manage_zip(item)
            elif item_extn == ".rar":
                manage_rar(item)
            elif item_extn == ".tar":
                manage_tar(item)
            elif item_extn == ".7z":
                manage_7z(item)
            elif item_extn == ".csv":
                manage_csv(item)
            else:
                #all other irrelevant files
                pass
        elif item.is_dir():
            manage_files(curr_path, item, irrelevant_files_dir, 
                        base_dir, processed_arch_dir, 
                        unprocessed_arch_dir, arch_extn)
            pass


    
    global global_counter
    # move all files to parent directory recursively
    for item in tqdm(parent_dir.rglob('*'), desc="Moving Files to parent_dir recursively..."):
        if item.exists():
            if item.is_file():
                if item.suffix.lower().strip() != ".csv":
                    dst_path = parent_dir / item.name
                    if dst_path.exists():
                        global_counter += 1
                        dst_path = parent_dir / f"{item.stem}_md{global_counter}{item.suffix}"
                    shutil.move(item, dst_path)
                else:
                    csv_parent = item.parent
                    print(csv_parent)
                    for file in csv_parent:
                        if file.suffix.lower().strip() == ".csv":
                            global_counter += 1
                            csv_dst_dir = base_dir / f"md{global_counter}"
                            os.makedirs(csv_dst_dir, exist_ok=True)
                            csv_dst_file = csv_dst_dir / file.name
                            shutil.move(file, csv_dst_file)

    # delete empty folders
    for item in tqdm(parent_dir.rglob('*'), desc="Handling directories..."):
        if item.is_dir():
            item_size = sum(file.stat().st_size for file in item.rglob('*') if file.is_file())
            item_empty = not any(item.iterdir())
            if item_empty or item_size==0:
                shutil.rmtree(item)
            
            
    
    #for item in tqdm(parent_dir.iterdir(), desc="Handling directories..."):
    #    if item.is_dir() and item.name != "00_new_folder":
    #        if is_base_dir(item):
    #            move_to_base_dir(item, base_dir)
    #            move_contents(item, parent_dir)
    #        else:
    #            move_contents(item, parent_dir)


    
        
    # now no directories are present at current location
    for item in tqdm(parent_dir.iterdir(), desc='Extracting archives...'):
        if item.is_file():
            # setting path to move the file after processing
            item_dir = item.parent
            item_name = item.name
            item_stem = item.stem    #item name without extension
            processed_arch = processed_arch_dir / item_name
            basic_zip_dest = base_dir / item_name
            unprocessed_arch = unprocessed_arch_dir / item_name
            irrelevant_file = irrelevant_files_dir / item_name
            
            if item.suffix in arch_extn:
                new_folder = parent_dir / "00_new_folder"
                os.makedirs(new_folder, exist_ok=True)
    
                try:
                    if zipfile.is_zipfile(item):
                        if is_base_zipfile(item):
                            shutil.move(item, basic_zip_dest)
                        else:
                            with zipfile.ZipFile(item, 'r') as zf:
                                zf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif rarfile.is_rarfile(item):
                        if is_base_rar(item):
                            item_name = f"{item_stem}.zip"
                            output_zip_path = item_dir / item_name
                            rar_to_zip(item, output_zip_path)
                            shutil.move(output_zip_path, basic_zip_dest)
                        else:
                            with rarfile.RarFile(item) as rf:
                                rf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif py7zr.is_7zfile(str(item)):
                        with py7zr.SevenZipFile(item, mode='r') as seven_zip:
                            seven_zip.extractall(path=new_folder)
                        shutil.move(item, processed_arch)
                        
                    elif tarfile.is_tarfile(item):
                        print(item)
                        with tarfile.open(item, 'r') as tar:
                            tar.extractall(new_folder)
                        shutil.move(item, processed_arch)
                        
                    else:
                        shutil.move(item, unprocessed_arch)
                        
                except Exception as e:
                    shutil.move(item, unprocessed_arch)
                    
            else:
                shutil.move(item, irrelevant_file)

    for item in parent_dir.iterdir():
        if item.is_dir() and item.name == "00_new_folder":
            aggregate_basic_zip(curr_path, item, irrelevant_files_dir, base_dir, processed_arch_dir, unprocessed_arch_dir, arch_extn)

In [9]:
def main():
    global global_counter
    global_counter = 100000
    curr_dir = Path.cwd()
    parent_dir = curr_dir / "01_parent_directory"
    irrelevant_files_dir = curr_dir / "02_irrelevant_files"
    unprocessed_archives_dir = curr_dir / "03_unprocessed_archives"
    processed_archives_dir = curr_dir / "04_processed_archives"
    base_dir = curr_dir / "05_base_folder"

    os.makedirs(irrelevant_files_dir, exist_ok=True)
    os.makedirs(unprocessed_archives_dir, exist_ok=True)
    os.makedirs(processed_archives_dir, exist_ok=True)
    os.makedirs(base_dir, exist_ok=True)

    arch_extn = ['.zip', '.rar', '.tar', '.gz', '.tgz', '.bz2', '.7z']
    
    aggregate_basic_zip(curr_dir, parent_dir, irrelevant_files_dir, 
                        base_dir, processed_archives_dir, 
                        unprocessed_archives_dir, arch_extn)

In [10]:
main()

Moving Files to parent_dir recursively...: 0it [00:00, ?it/s]

D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\01_parent_directory


TypeError: 'WindowsPath' object is not iterable